In [1]:
!pip install konlpy

  Using cached konlpy-0.6.0-py2.py3-none-any.whl.metadata (1.9 kB)
Using cached konlpy-0.6.0-py2.py3-none-any.whl (19.4 MB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 18.4 MB/s  0:00:00

   ---------------------------------------- 0/3 [lxml]
   ---------------------------------------- 0/3 [lxml]
   ---------------------------------------- 0/3 [lxml]
   ---------------------------------------- 0/3 [lxml]
   ---------------------------------------- 0/3 [lxml]
   ------------- -------------------------- 1/3 [JPype1]
   ------------- -------------------------- 1/3 [JPype1]
   -------------------------- ------------- 2/3 [konlpy]
   -------------------------- ------------- 2/3 [konlpy]
   -------------------------- ------------- 2/3 [konlpy]
   ---------------------------------------- 3/3 [konlpy]



In [2]:
import re
from konlpy.tag import Okt
from collections import Counter

In [5]:
text = "임금님 귀는 당나귀 귀! 임금님 귀는 당나귀 귀! 실컷~ 소리치고 나니 속이 확 뚫려 살 것 같았어."
text

'임금님 귀는 당나귀 귀! 임금님 귀는 당나귀 귀! 실컷~ 소리치고 나니 속이 확 뚫려 살 것 같았어.'

In [6]:
reg = re.compile("[^가-힣 ]")
text = reg.sub('', text)
text

'임금님 귀는 당나귀 귀 임금님 귀는 당나귀 귀 실컷 소리치고 나니 속이 확 뚫려 살 것 같았어'

In [7]:
%%time
okt=Okt()
tokens = okt.morphs(text)
print(tokens)

['임금님', '귀', '는', '당나귀', '귀', '임금님', '귀', '는', '당나귀', '귀', '실컷', '소리', '치고', '나니', '속이', '확', '뚫려', '살', '것', '같았어']
CPU times: total: 11.5 s
Wall time: 5.86 s


In [8]:
vocab = Counter(tokens)
print(vocab)

Counter({'귀': 4, '임금님': 2, '는': 2, '당나귀': 2, '실컷': 1, '소리': 1, '치고': 1, '나니': 1, '속이': 1, '확': 1, '뚫려': 1, '살': 1, '것': 1, '같았어': 1})


In [9]:
vocab['임금님']

2

In [10]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
print(vocab)

[('귀', 4), ('임금님', 2), ('는', 2), ('당나귀', 2), ('실컷', 1)]


In [11]:
word2idx={word[0] : index+1 for index, word in enumerate(vocab)}
print(word2idx)

{'귀': 1, '임금님': 2, '는': 3, '당나귀': 4, '실컷': 5}


In [13]:
# 원-핫 인코딩
def one_hot_encoding(word, word2index):
       one_hot_vector = [0]*(len(word2index))
       index = word2index[word]
       one_hot_vector[index-1] = 1
       return one_hot_vector

In [14]:
one_hot_encoding("임금님", word2idx)

[0, 1, 0, 0, 0]

### 파이토치를 통한 원-핫 인코딩

In [16]:
import torch
import torch.nn.functional as F
from collections import Counter

In [17]:
text = [['강아지', '고양이', '강아지'],['애교', '고양이'], ['컴퓨터', '노트북']]
text

[['강아지', '고양이', '강아지'], ['애교', '고양이'], ['컴퓨터', '노트북']]

In [18]:
counter = Counter(word for sentence in text for word in sentence) # 단어 등장 빈도 계산
word_index = {word: i+1 for i, (word, _) in enumerate(counter.most_common())}
word_index["<PAD>"] = 0   # OOV(Out-of-Vocabulary) 처리를 위해 기본 인덱스 설정

print(word_index)

{'강아지': 1, '고양이': 2, '애교': 3, '컴퓨터': 4, '노트북': 5, '<PAD>': 0}


In [19]:
vocab_size = len(word_index) # 단어 사전 크기

In [20]:
sub_text = ['강아지', '고양이', '강아지', '컴퓨터']
encoded = [word_index.get(word, 0) for word in sub_text]  # OOV 단어는 0으로 처리
print(encoded)

[1, 2, 1, 4]


In [22]:
# 원-핫 텐서 변환
encoded_tensor = torch.tensor(encoded, dtype=torch.long) # PyTorch 텐서 변환
one_hot = F.one_hot(encoded_tensor, num_classes=vocab_size)
print(one_hot)

tensor([[0, 1, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0]])


## Sparse vector(희소 벡터)의 문제점

1. 차원의 저주
2. 유사도 측정 불가

## Word Embedding

- 한 단어를 길이가 비교적 짧은 **밀집 벡터**로 나타냄
- 이 밀집 벡터는 단어가 갖는 의미나 단어 간의 관계 등을 내포함

### Word2Vec

In [1]:
import nltk
nltk.download('abc')
nltk.download('punkt')

[nltk_data] Downloading package abc to
[nltk_data]     C:\Users\heeji\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\abc.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\heeji\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [5]:
from nltk.corpus import abc
corpus = abc.sents()

print(corpus[:3])
print('코퍼스의 크기 :',len(corpus))

[['PM', 'denies', 'knowledge', 'of', 'AWB', 'kickbacks', 'The', 'Prime', 'Minister', 'has', 'denied', 'he', 'knew', 'AWB', 'was', 'paying', 'kickbacks', 'to', 'Iraq', 'despite', 'writing', 'to', 'the', 'wheat', 'exporter', 'asking', 'to', 'be', 'kept', 'fully', 'informed', 'on', 'Iraq', 'wheat', 'sales', '.'], ['Letters', 'from', 'John', 'Howard', 'and', 'Deputy', 'Prime', 'Minister', 'Mark', 'Vaile', 'to', 'AWB', 'have', 'been', 'released', 'by', 'the', 'Cole', 'inquiry', 'into', 'the', 'oil', 'for', 'food', 'program', '.'], ['In', 'one', 'of', 'the', 'letters', 'Mr', 'Howard', 'asks', 'AWB', 'managing', 'director', 'Andrew', 'Lindberg', 'to', 'remain', 'in', 'close', 'contact', 'with', 'the', 'Government', 'on', 'Iraq', 'wheat', 'sales', '.']]
코퍼스의 크기 : 29059


In [6]:
%%time
from gensim.models import Word2Vec

model = Word2Vec(sentences = corpus, vector_size = 100, window = 5, min_count = 5, workers = 4, sg = 0)
print("모델 학습 완료!")

모델 학습 완료!
CPU times: total: 13.5 s
Wall time: 24.4 s


- vector size = 학습 후 임베딩 벡터의 차원
- window = 컨텍스트 윈도우 크기
- min_count = 단어 최소 빈도수 제한 (빈도가 적은 단어들은 학습하지 않아요.)
- workers = 학습을 위한 프로세스 수
- sg = 0은 CBoW, 1은 Skip-gram

In [7]:
model_result = model.wv.most_similar("man")
print(model_result)

[('woman', 0.9233695268630981), ('skull', 0.9111703038215637), ('Bang', 0.905643105506897), ('asteroid', 0.9050782918930054), ('third', 0.9020301699638367), ('baby', 0.8994044661521912), ('dog', 0.8985670208930969), ('bought', 0.8975346088409424), ('rally', 0.8912985324859619), ('disc', 0.8890827894210815)]


In [11]:
# 모델 저장 및 로드
from gensim.models import KeyedVectors

model.wv.save_word2vec_format('models/w2v')
loaded_model = KeyedVectors.load_word2vec_format("models/w2v")
print("모델  load 완료!")

모델  load 완료!


In [12]:
model_result = loaded_model.most_similar("man")
print(model_result)

[('woman', 0.9233695268630981), ('skull', 0.9111703038215637), ('Bang', 0.905643105506897), ('asteroid', 0.9050782918930054), ('third', 0.9020301699638367), ('baby', 0.8994044661521912), ('dog', 0.8985670208930969), ('bought', 0.8975346088409424), ('rally', 0.8912985324859619), ('disc', 0.8890827894210815)]


In [13]:
# OOV 문제
loaded_model.most_similar('overacting')

KeyError: "Key 'overacting' not present in vocabulary"

In [14]:
# OOV 문제(오타)
loaded_model.most_similar('memorry')

KeyError: "Key 'memorry' not present in vocabulary"

In [15]:
!python -m gensim.scripts.word2vec2tensor --input models/w2v --output models/w2v

2026-08-10 14:28:24,952 - word2vec2tensor - INFO - running C:\Users\heeji\anaconda3\envs\AIFFEL_quest\Lib\site-packages\gensim\scripts\word2vec2tensor.py --input models/w2v --output models/w2v
2026-08-10 14:28:24,952 - keyedvectors - INFO - loading projection weights from models/w2v
2026-08-10 14:28:25,803 - utils - INFO - KeyedVectors lifecycle event {'msg': 'loaded (10363, 100) matrix of type float32 from models/w2v', 'binary': False, 'encoding': 'utf8', 'datetime': '2026-08-10T14:28:25.720873', 'gensim': '4.3.2', 'python': '3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'event': 'load_word2vec_format'}
2026-08-10 14:28:26,632 - word2vec2tensor - INFO - 2D tensor file saved to models/w2v_tensor.tsv
2026-08-10 14:28:26,633 - word2vec2tensor - INFO - Tensor metadata file saved to models/w2v_metadata.tsv
2026-08-10 14:28:26,633 - word2vec2tensor - INFO - finished running word2vec2tensor.py


### FaskText

In [16]:
%%time
from gensim.models import FastText
fasttext_model = FastText(corpus, window=5, min_count=5, workers=4, sg=1)
print("FastText 학습 완료!")

FastText 학습 완료!
CPU times: total: 41.7 s
Wall time: 20.3 s


In [17]:
fasttext_model.wv.most_similar('overacting')

[('extracting', 0.941209077835083),
 ('fluctuating', 0.9343767762184143),
 ('resolving', 0.9330555200576782),
 ('malting', 0.9329907894134521),
 ('shooting', 0.9322556257247925),
 ('overwhelming', 0.9322171211242676),
 ('emptying', 0.9319466352462769),
 ('lifting', 0.9303894639015198),
 ('attracting', 0.9295125603675842),
 ('mounting', 0.9287572503089905)]

In [18]:
fasttext_model.wv.most_similar('memoryy')

[('memory', 0.947331964969635),
 ('mechanism', 0.8658248782157898),
 ('mechanisms', 0.8651583790779114),
 ('basic', 0.8602371215820312),
 ('musical', 0.8569340109825134),
 ('technical', 0.8549808859825134),
 ('video', 0.8504824638366699),
 ('mechanical', 0.8444274067878723),
 ('intelligence', 0.841383695602417),
 ('duplicate', 0.8384506702423096)]

### GloVe

In [19]:
import gensim.downloader as api
glove_model = api.load("glove-wiki-gigaword-50")  # glove vectors 다운로드
glove_model.most_similar("dog")  # 'dog'과 비슷한 단어 찾기

[==================================================] 100.0% 66.0/66.0MB downloaded


[('cat', 0.9218004941940308),
 ('dogs', 0.8513158559799194),
 ('horse', 0.7907583713531494),
 ('puppy', 0.7754920721054077),
 ('pet', 0.7724708318710327),
 ('rabbit', 0.7720814347267151),
 ('pig', 0.7490062117576599),
 ('snake', 0.7399188876152039),
 ('baby', 0.7395570278167725),
 ('bite', 0.7387937307357788)]

In [20]:
glove_model.most_similar('overacting')

[('impudence', 0.7842012047767639),
 ('puerile', 0.7816032767295837),
 ('winningly', 0.7644237875938416),
 ('grossness', 0.7576098442077637),
 ('deconstructions', 0.748936653137207),
 ('over-the-top', 0.7460805773735046),
 ('buffoonery', 0.746045708656311),
 ('impetuosity', 0.7415392398834229),
 ('sophomoric', 0.736961841583252),
 ('zaniness', 0.7353197336196899)]

In [21]:
# OOV 문제(오타)
glove_model.most_similar('memoryy')

KeyError: "Key 'memoryy' not present in vocabulary"